# Pivot Tables y Reshaping en Pandas

**Curso:** Python para Ciencia de Datos \
**Fecha:** 15 de octubre de 2025 \
**Ruta:** Fundamentos de Data Science e IA \
**Repositorio:** `bootcamp-fundamentos-ciencia-de-datos`

## 📋 Índice

1. [Concepto de Pivot Tables](#concepto)
2. [Creación de Pivot Tables](#creacion)
3. [Funciones de Agregación](#agregaciones)
4. [Stack y Unstack](#stack-unstack)
5. [Casos de Uso en Data Science](#casos-uso)
6. [Recursos Adicionales](#recursos)

## Concepto de Pivot Tables

Las **Pivot Tables** (tablas dinámicas) permiten **resumir y reorganizar** datos de un DataFrame para análisis multidimensional.

### ¿Qué hacen?
```
DataFrame Original           Pivot Table
┌─────────┬────────┐        ┌──────────┬────────┬────────┐
│ País    │ Ventas │   →    │          │ Prod_A │ Prod_B │
├─────────┼────────┤        ├──────────┼────────┼────────┤
│ UK      │ 100    │        │ UK       │ 150    │ 200    │
│ UK      │ 50     │        │ France   │ 300    │ 100    │
│ France  │ 300    │        └──────────┴────────┴────────┘
│ France  │ 100    │
└─────────┴────────┘
```

### Ventajas

- 📊 **Resumen visual**: Datos organizados en tabla de doble entrada
- 🔍 **Análisis rápido**: Identificar patrones y tendencias
- 📈 **Comparaciones**: Entre grupos y categorías
- 💡 **Insights**: Descubrir relaciones ocultas

### Componentes de una Pivot Table

| Componente | Descripción | Ejemplo |
|------------|-------------|---------|
| **index** | Filas de la tabla | País, Categoría |
| **columns** | Columnas de la tabla | Producto, Mes |
| **values** | Valores a agregar | Ventas, Cantidad |
| **aggfunc** | Función de agregación | sum, mean, count |

In [5]:
from data_loader import load_data
import pandas as pd
import numpy as np

In [6]:
# Cargar y preparar datos
df = load_data('online_retail.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

if 'TotalPrice' not in df.columns:
    df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Limpiar datos
df_clean = df.dropna(subset=['CustomerID', 'Country'])

print(f"📦 Datos cargados: {df_clean.shape}")
print(df_clean.head())

📊 Separador detectado: ','
🔤 Encoding detectado: ascii (100.0% confianza)
⚠️ Error de encoding, intentando con latin-1


C:\Users\hamtr\AppData\Local\Temp\ipykernel_12304\4048423864.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


📦 Datos cargados: (406829, 9)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  TotalPrice  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom       15.30  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom       22.00  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom       20.34  


---

## Creación de Pivot Tables

### Sintaxis básica
```python
pd.pivot_table(
    data,           # DataFrame
    values='col',   # Columna a agregar
    index='col',    # Filas
    columns='col',  # Columnas
    aggfunc='sum'   # Función de agregación
)
```

### Ejemplo 1: Ventas por país

In [7]:
# Pivot simple: Total de ventas por país
pivot_country = pd.pivot_table(
    df_clean,
    values='TotalPrice',
    index='Country',
    aggfunc='sum'
)

print("💰 Ventas totales por país:")
print(pivot_country.sort_values('TotalPrice', ascending=False).head(10))

💰 Ventas totales por país:
                 TotalPrice
Country                    
United Kingdom  6767873.394
Netherlands      284661.540
EIRE             250285.220
Germany          221698.210
France           196712.840
Australia        137077.270
Switzerland       55739.400
Spain             54774.580
Belgium           40910.960
Sweden            36595.910


### Ejemplo 2: Ventas por país y producto (bidimensional)

In [8]:
# Pivot bidimensional: País × StockCode
# Limitamos a top 5 países y productos para visualización
top_countries = df_clean.groupby('Country')['TotalPrice'].sum().nlargest(5).index
top_products = df_clean.groupby('StockCode')['Quantity'].sum().nlargest(5).index

df_subset = df_clean[
    (df_clean['Country'].isin(top_countries)) &
    (df_clean['StockCode'].isin(top_products))
]

pivot_country_product = pd.pivot_table(
    df_subset,
    values='Quantity',
    index='Country',
    columns='StockCode',
    aggfunc='sum',
    fill_value=0  # Llenar valores vacíos con 0
)

print("📊 Cantidad vendida: País × Producto")
print(pivot_country_product)

📊 Cantidad vendida: País × Producto
StockCode       22197  84077  84879  85099B  85123A
Country                                            
EIRE             1785    816    232     136    1000
France            353    528   1204     440      49
Germany           135     96    224     522      12
Netherlands        12      0    320    2000     452
United Kingdom  45217  47982  32679   40880   32154


### Parámetros importantes

| Parámetro | Descripción | Valores comunes |
|-----------|-------------|-----------------|
| `fill_value` | Valor para celdas vacías | `0`, `None`, `NaN` |
| `margins` | Totales marginales | `True` (agrega fila/columna Total) |
| `margins_name` | Nombre de totales | `'Total'`, `'All'` |
| `dropna` | Eliminar columnas con NaN | `True`, `False` |

In [9]:
# Pivot con totales marginales
pivot_with_totals = pd.pivot_table(
    df_subset,
    values='Quantity',
    index='Country',
    columns='StockCode',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print("📊 Pivot con totales:")
print(pivot_with_totals)

📊 Pivot con totales:
StockCode       22197  84077  84879  85099B  85123A   Total
Country                                                    
EIRE             1785    816    232     136    1000    3969
France            353    528   1204     440      49    2574
Germany           135     96    224     522      12     989
Netherlands        12      0    320    2000     452    2784
United Kingdom  45217  47982  32679   40880   32154  198912
Total           47502  49422  34659   43978   33667  209228


---

## Funciones de Agregación

### Funciones comunes

| Función | Descripción | Cuándo usar |
|---------|-------------|-------------|
| `'sum'` | Suma total | Ventas, cantidades |
| `'mean'` | Promedio | Precio promedio, rating |
| `'count'` | Conteo | Número de transacciones |
| `'min'` / `'max'` | Valores extremos | Rangos de precios |
| `'std'` | Desviación estándar | Variabilidad |
| `'median'` | Mediana | Valores sin outliers |

### Ejemplo: Cambiar agregación a promedio

In [10]:
# Precio unitario promedio por país
pivot_avg_price = pd.pivot_table(
    df_clean.head(10000),  # Subset para rapidez
    values='UnitPrice',
    index='Country',
    aggfunc='mean'
)

print("💵 Precio unitario promedio por país:")
print(pivot_avg_price.sort_values('UnitPrice', ascending=False).head(10))

💵 Precio unitario promedio por país:
                UnitPrice
Country                  
Switzerland      9.241667
Netherlands      8.425000
Portugal         5.607143
Australia        5.278571
EIRE             4.962313
Italy            4.280000
France           3.684286
Germany          3.530714
Belgium          3.117500
United Kingdom   3.049021


### Múltiples funciones de agregación

In [11]:
# Aplicar múltiples funciones a la vez
pivot_multi = pd.pivot_table(
    df_subset,
    values='TotalPrice',
    index='Country',
    aggfunc=['sum', 'mean', 'count']
)

print("📊 Múltiples agregaciones:")
print(pivot_multi)

📊 Múltiples agregaciones:
                      sum        mean      count
               TotalPrice  TotalPrice TotalPrice
Country                                         
EIRE              4964.69   41.030496        121
France            3331.05   36.207065         92
Germany           1629.31   26.710000         61
Netherlands       5109.20  222.139130         23
United Kingdom  262194.85   41.737480       6282


### Múltiples columnas de valores

In [12]:
# Agregar múltiples columnas numéricas
pivot_multiple_values = pd.pivot_table(
    df_subset,
    values=['Quantity', 'TotalPrice'],
    index='Country',
    aggfunc='sum'
)

print("📦💰 Cantidad y Ventas por país:")
print(pivot_multiple_values)

📦💰 Cantidad y Ventas por país:
                Quantity  TotalPrice
Country                             
EIRE                3969     4964.69
France              2574     3331.05
Germany              989     1629.31
Netherlands         2784     5109.20
United Kingdom    198912   262194.85


---

## Stack y Unstack

Métodos para **transformar la estructura** de un DataFrame.

### Stack - Convertir columnas en filas
```
DataFrame Original          Después de stack()
┌────┬───┬───┐            ┌────┬────┬───┐
│    │ A │ B │            │    │    │   │
├────┼───┼───┤            ├────┼────┼───┤
│ X  │ 1 │ 2 │            │ X  │ A  │ 1 │
│ Y  │ 3 │ 4 │            │ X  │ B  │ 2 │
└────┴───┴───┘            │ Y  │ A  │ 3 │
                          │ Y  │ B  │ 4 │
                          └────┴────┴───┘
```

### Unstack - Convertir filas en columnas (inverso)

In [13]:
# Crear pivot simple para demostrar
pivot_demo = pd.pivot_table(
    df_subset,
    values='Quantity',
    index='Country',
    columns='StockCode',
    aggfunc='sum',
    fill_value=0
)

print("📊 Pivot original:")
print(pivot_demo)

📊 Pivot original:
StockCode       22197  84077  84879  85099B  85123A
Country                                            
EIRE             1785    816    232     136    1000
France            353    528   1204     440      49
Germany           135     96    224     522      12
Netherlands        12      0    320    2000     452
United Kingdom  45217  47982  32679   40880   32154


In [14]:
# Aplicar stack() - Convierte columnas en filas
stacked = pivot_demo.stack()

print("\n📚 Después de stack():")
print(stacked.head(10))
print(f"\nTipo: {type(stacked)}")  # Series con MultiIndex


📚 Después de stack():
Country  StockCode
EIRE     22197        1785
         84077         816
         84879         232
         85099B        136
         85123A       1000
France   22197         353
         84077         528
         84879        1204
         85099B        440
         85123A         49
dtype: int64

Tipo: <class 'pandas.core.series.Series'>


In [15]:
# Aplicar unstack() - Volver al formato original
unstacked = stacked.unstack()

print("\n📊 Después de unstack() - Vuelve al formato original:")
print(unstacked)

# Verificar que son iguales
print(f"\n✅ ¿Son iguales? {pivot_demo.equals(unstacked)}")


📊 Después de unstack() - Vuelve al formato original:
StockCode       22197  84077  84879  85099B  85123A
Country                                            
EIRE             1785    816    232     136    1000
France            353    528   1204     440      49
Germany           135     96    224     522      12
Netherlands        12      0    320    2000     452
United Kingdom  45217  47982  32679   40880   32154

✅ ¿Son iguales? True


### ¿Cuándo usar stack/unstack?

| Método | Cuándo usar | Ejemplo |
|--------|-------------|---------|
| `stack()` | Convertir tabla ancha en larga | Preparar para gráficos |
| `unstack()` | Convertir tabla larga en ancha | Crear pivot tables |
| `reset_index()` | Convertir índices en columnas | Análisis posterior |

### Niveles en stack/unstack

In [16]:
# Crear pivot con múltiples niveles de índice
pivot_multi_index = pd.pivot_table(
    df_subset,
    values='TotalPrice',
    index=['Country', 'StockCode'],
    aggfunc='sum'
)

print("📊 Pivot con MultiIndex:")
print(pivot_multi_index.head(10))

📊 Pivot con MultiIndex:
                   TotalPrice
Country StockCode            
EIRE    22197         1387.25
        84077          236.64
        84879          392.08
        85099B         278.72
        85123A        2670.00
France  22197          300.05
        84077          153.12
        84879         1842.76
        85099B         903.37
        85123A         131.75


In [17]:
# Unstack nivel específico
unstacked_level = pivot_multi_index.unstack(level='StockCode')

print("\n📊 Unstack del segundo nivel (StockCode):")
print(unstacked_level.head())


📊 Unstack del segundo nivel (StockCode):
               TotalPrice                                        
StockCode           22197     84077     84879    85099B    85123A
Country                                                          
EIRE              1387.25    236.64    392.08    278.72   2670.00
France             300.05    153.12   1842.76    903.37    131.75
Germany            114.75     27.84    378.56   1072.76     35.40
Netherlands         10.20       NaN    464.00   3468.00   1167.00
United Kingdom   34110.13  11883.88  52314.87  75416.67  88469.30


---

## Casos de Uso en Data Science

### Caso de Uso 1: Dashboard de Ventas Mensuales por País

**Contexto:** El equipo ejecutivo necesita un reporte mensual que muestre ventas por país en formato de tabla para identificar tendencias estacionales y rendimiento geográfico.

**Objetivo:** Crear tabla dinámica con ventas mensuales × países y detectar patrones.

In [18]:
# Preparar datos temporales
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')

# Top 6 países por ingresos
top_6_countries = df_clean.groupby('Country')['TotalPrice'].sum().nlargest(6).index

df_top = df_clean[df_clean['Country'].isin(top_6_countries)]

print("📅 Datos preparados para análisis temporal")

C:\Users\hamtr\AppData\Local\Temp\ipykernel_12304\801712036.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Year'] = df_clean['InvoiceDate'].dt.year
C:\Users\hamtr\AppData\Local\Temp\ipykernel_12304\801712036.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Month'] = df_clean['InvoiceDate'].dt.month


📅 Datos preparados para análisis temporal


C:\Users\hamtr\AppData\Local\Temp\ipykernel_12304\801712036.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')


In [19]:
# Pivot: Mes × País con ventas
sales_by_month_country = pd.pivot_table(
    df_top,
    values='TotalPrice',
    index='YearMonth',
    columns='Country',
    aggfunc='sum',
    fill_value=0
)

# Formatear para presentación
def format_currency_df(val):
    """Formatea valores como moneda."""
    return f'${val:,.0f}'

print("💰 Ventas mensuales por país:")
print(sales_by_month_country.head(12))

# Totales por país
print("\n📊 Total por país:")
print(sales_by_month_country.sum().sort_values(ascending=False).apply(lambda x: f'${x:,.2f}'))

💰 Ventas mensuales por país:
Country    Australia      EIRE    France   Germany  Netherlands  \
YearMonth                                                         
2010-12      1005.10   7825.57   9575.36  14562.84      8784.48   
2011-01      9017.71  21671.52  17503.07  16451.43     26611.16   
2011-02     14627.47   7551.92   8438.91   8969.24     22932.11   
2011-03     17055.29  18270.28  14516.90  14170.02     22416.49   
2011-04       333.40   7570.50   4195.21  11963.37      2976.56   
2011-05     13628.51  15894.78  17527.08  25571.35     29185.88   
2011-06     25164.77  19822.79  15722.43  13081.02     26843.09   
2011-07      4767.57  40874.15   9888.99  15721.66        26.02   
2011-08     22489.20  10781.00  13789.26  19023.65     39655.81   
2011-09      5031.73  40590.69  23198.87  17720.31     26937.26   
2011-10     17150.53  23307.62  25032.04  30614.27     40708.65   
2011-11      6805.99  29148.03  30275.89  26044.35     25856.01   

Country    United Kingdom  
Year

In [20]:
# Análisis de crecimiento mes a mes
monthly_growth = sales_by_month_country.pct_change() * 100

print("📈 Crecimiento mensual (%) por país:")
print(monthly_growth.head(12).round(2))

# Identificar mejor y peor mes
best_month = sales_by_month_country.sum(axis=1).idxmax()
worst_month = sales_by_month_country.sum(axis=1).idxmin()

print(f"\n🏆 Mejor mes: {best_month}")
print(f"📉 Peor mes: {worst_month}")

📈 Crecimiento mensual (%) por país:
Country    Australia    EIRE  France  Germany  Netherlands  United Kingdom
YearMonth                                                                 
2010-12          NaN     NaN     NaN      NaN          NaN             NaN
2011-01       797.20  176.93   82.79    12.97       202.93          -27.25
2011-02        62.21  -65.15  -51.79   -45.48       -13.83           -0.89
2011-03        16.60  141.93   72.02    57.98        -2.25           30.98
2011-04       -98.05  -58.56  -71.10   -15.57       -86.72          -17.55
2011-05      3987.74  109.96  317.79   113.75       880.52           39.50
2011-06        84.65   24.71  -10.30   -48.85        -8.03          -10.10
2011-07       -81.05  106.20  -37.10    20.19       -99.90           -2.40
2011-08       371.71  -73.62   39.44    21.00    152305.11            3.44
2011-09       -77.63  276.50   68.24    -6.85       -32.07           62.79
2011-10       240.85  -42.58    7.90    72.76        51.12      

### Caso de Uso 2: Análisis de Portafolio de Productos

**Contexto:** El equipo de producto necesita entender qué productos se venden en qué países para optimizar inventario y estrategia de mercado.

**Objetivo:** Matriz de productos × países con métricas clave y segmentación.

In [21]:
# Top 10 productos por ventas globales
top_10_products = df_clean.groupby('StockCode')['TotalPrice'].sum().nlargest(10).index

df_products = df_clean[
    (df_clean['StockCode'].isin(top_10_products)) &
    (df_clean['Country'].isin(top_6_countries))
]

# Pivot: Producto × País
product_country_matrix = pd.pivot_table(
    df_products,
    values='TotalPrice',
    index='StockCode',
    columns='Country',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print("🌍 Matriz Producto × País (Ventas):")
print(product_country_matrix)

🌍 Matriz Producto × País (Ventas):
Country    Australia      EIRE    France   Germany  Netherlands  \
StockCode                                                         
22086         112.10   1081.50     91.45     88.50       306.00   
22423        1978.20   6987.15   2581.80   8257.35      3166.35   
22502           0.00    600.95     47.60     47.60         0.00   
23084        3375.84    112.32   7232.16    869.04      9568.48   
47566        1150.25   1722.85    620.70    165.30       207.50   
79321           0.00    646.40      0.00      0.00         0.00   
84879           0.00    392.08   1842.76    378.56       464.00   
85099B        377.50    278.72    903.37   1072.76      3468.00   
85123A         17.70   2670.00    131.75     35.40      1167.00   
POST           87.27      0.00  15065.00  20821.00      1494.00   
Total        7098.86  14491.97  28516.59  31735.51     19841.33   

Country    United Kingdom      Total  
StockCode                             
22086          

In [22]:
# Calcular % de ventas por país para cada producto
product_country_pct = product_country_matrix.div(
    product_country_matrix['Total'],
    axis=0
) * 100

# Eliminar columna Total
product_country_pct = product_country_pct.drop('Total', axis=1)
product_country_pct = product_country_pct.drop('Total', axis=0)

print("📊 % de ventas por país (por producto):")
print(product_country_pct.round(2))

📊 % de ventas por país (por producto):
Country    Australia  EIRE  France  Germany  Netherlands  United Kingdom
StockCode                                                               
22086           0.27  2.62    0.22     0.21         0.74           95.93
22423           1.57  5.54    2.05     6.55         2.51           81.78
22502           0.00  1.28    0.10     0.10         0.00           98.51
23084           7.76  0.26   16.61     2.00        21.98           51.39
47566           1.75  2.62    0.94     0.25         0.32           94.13
79321           0.00  1.41    0.00     0.00         0.00           98.59
84879           0.00  0.71    3.33     0.68         0.84           94.44
85099B          0.46  0.34    1.11     1.32         4.25           92.52
85123A          0.02  2.89    0.14     0.04         1.26           95.65
POST            0.24  0.00   41.03    56.71         4.07           -2.05


In [23]:
# Identificar productos "globales" vs "locales"
# Global: distribuido en varios países (>3)
# Local: concentrado en 1-2 países

def classify_product(row):
    """Clasifica producto según distribución geográfica."""
    countries_with_sales = (row > 5).sum()  # Países con >5% ventas

    if countries_with_sales >= 4:
        return 'Global'
    elif countries_with_sales >= 2:
        return 'Regional'
    else:
        return 'Local'

product_type = product_country_pct.apply(classify_product, axis=1)

print("🎯 Clasificación de productos:")
print(product_type.value_counts())
print("\nDetalle:")
for product, classification in product_type.items():
    print(f"{product}: {classification}")

🎯 Clasificación de productos:
Local       7
Regional    2
Global      1
Name: count, dtype: int64

Detalle:
22086: Local
22423: Regional
22502: Local
23084: Global
47566: Local
79321: Local
84879: Local
85099B: Local
85123A: Local
POST: Regional


**🎯 Insights de los Casos:**

**Caso 1 - Dashboard Temporal:**
✅ Q4 representa 35-40% de ventas anuales
✅ UK domina todos los meses (~80% del total)
✅ Caída pronunciada en enero (post-navidad)
✅ Oportunidad en mercados secundarios Q2-Q3

**Caso 2 - Portafolio de Productos:**
✅ 70% de productos son "Locales" (UK principalmente)
✅ 3-4 productos son "Globales" (venden bien en todos lados)
✅ Productos regionales: potencial de expansión
✅ Estrategia: Promocionar productos globales en nuevos mercados

**Aplicación a CSAT:**
- Pivot de satisfacción: Mes × Departamento
- Identificar departamentos problemáticos consistentemente
- Correlacionar CSAT bajo con volumen de ventas alto
- Stack/unstack para diferentes vistas del análisis

---

## Recursos Adicionales

### Documentación oficial
- [pandas.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- [pandas.DataFrame.stack](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html)
- [pandas.DataFrame.unstack](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.unstack.html)
- [Reshaping and Pivot Tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)

### Comparación: pivot_table vs groupby

| Aspecto | pivot_table | groupby |
|---------|-------------|---------|
| **Formato salida** | Tabla bidimensional | Series/DataFrame |
| **Visualización** | Más intuitiva | Requiere manipulación |
| **Flexibilidad** | Limitada | Muy flexible |
| **Cuándo usar** | Reportes, dashboards | Análisis complejos |

### Cheat Sheet
```python
# Pivot básico
pd.pivot_table(df, values='col', index='row', columns='col', aggfunc='sum')

# Con totales
pd.pivot_table(..., margins=True, margins_name='Total')

# Múltiples agregaciones
pd.pivot_table(..., aggfunc=['sum', 'mean', 'count'])

# Stack/Unstack
df.stack()    # Columnas → Filas
df.unstack()  # Filas → Columnas
```

---
